# 14 - Score a User-Provided Message or Conversation

Use this notebook to score one sentence or a conversation text file with the saved Stage 1 and Stage 2 models. For text files, use one message per line and optionally prefix each line with `Speaker:`.

Stage 1 models predict one of the coded stages (200/600/900). These models were trained only on predatory coded messages, so their outputs are stage estimates, not safe/unsafe classifications. Stage 2 models estimate PAN-12 conversation-level risk.

The color is binary and is based only on the two available Stage 2 models: red if either reaches its saved validation cutoff; otherwise green. Green means only “not flagged by these models,” not safe. The cutoffs were selected on a validation split from PAN-12 and have not been calibrated for real-world conversations.

Keep all source data local. The notebook does not save supplied message text or scores to files. Scores are displayed in notebook outputs. The sentence-transformer may need to download its model once if it is not already cached.

In [1]:
from pathlib import Path
import re
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

STAGE_ORDER = ["200", "600", "900"]
WORKSPACE = Path.cwd()
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR = WORKSPACE / "reports"
class LexiconTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lexicon_categories):
        self.lexicon_categories = lexicon_categories

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        word_pattern = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
        feature_rows = []
        for text in X:
            normalized = re.sub(r"\s+", " ", str(text).lower()).strip()
            token_count = max(1, len(word_pattern.findall(normalized)))
            row = []
            padded = f" {normalized} "
            for entries in self.lexicon_categories.values():
                count = sum(padded.count(f" {entry.strip().lower()} ") for entry in entries if entry.strip())
                row.extend([count, count / token_count])
            feature_rows.append(row)
        return np.asarray(feature_rows, dtype=float)

print("Project directory:", WORKSPACE)
print("Define LexiconTransformer before loading the saved lexicon model so its pickle can resolve the class.")

Project directory: f:\NLP Project Final
Define LexiconTransformer before loading the saved lexicon model so its pickle can resolve the class.


In [2]:
required_artifacts = [
    MODEL_DIR / "stage1_tfidf_logistic.joblib",
    MODEL_DIR / "stage1_tfidf_lexicon_logistic.joblib",
    MODEL_DIR / "stage1_embedding_logistic.joblib",
    MODEL_DIR / "stage2_logistic_all.joblib",
    MODEL_DIR / "stage2_lightgbm_all_numeric.joblib",
    REPORT_DIR / "stage1_transition_matrix.csv",
    REPORT_DIR / "stage2_ablation_thresholds.json",
    REPORT_DIR / "stage2_lightgbm_thresholds.json",
]
missing = [path for path in required_artifacts if not path.exists()]
if missing:
    raise FileNotFoundError("Run the preceding training notebooks first. Missing: " + ", ".join(str(path) for path in missing))

stage1_models = {
    "TF-IDF baseline": joblib.load(MODEL_DIR / "stage1_tfidf_logistic.joblib"),
    "TF-IDF + lexicons": joblib.load(MODEL_DIR / "stage1_tfidf_lexicon_logistic.joblib"),
}
embedding_classifier = joblib.load(MODEL_DIR / "stage1_embedding_logistic.joblib")
stage2_models = {
    "Stage 2 logistic (text + numeric)": joblib.load(MODEL_DIR / "stage2_logistic_all.joblib"),
    "Stage 2 LightGBM (numeric)": joblib.load(MODEL_DIR / "stage2_lightgbm_all_numeric.joblib"),
}

logistic_threshold = json.loads((REPORT_DIR / "stage2_ablation_thresholds.json").read_text(encoding="utf-8"))["all"]
lightgbm_threshold = json.loads((REPORT_DIR / "stage2_lightgbm_thresholds.json").read_text(encoding="utf-8"))["all_numeric"]
stage2_thresholds = {
    "Stage 2 logistic (text + numeric)": logistic_threshold,
    "Stage 2 LightGBM (numeric)": lightgbm_threshold,
}

transition_frame = pd.read_csv(REPORT_DIR / "stage1_transition_matrix.csv", index_col=0)
transition_frame.index = transition_frame.index.map(str)
transition_frame.columns = transition_frame.columns.map(str)
transition_matrix = transition_frame.loc[STAGE_ORDER, STAGE_ORDER].to_numpy(dtype=float)

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Install sentence-transformers to use the saved Stage 1 embedding model.") from exc

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Loaded 3 Stage 1 models and 2 Stage 2 models.")
print("Stage 2 validation cutoffs:", stage2_thresholds)

c:\Users\Kruti\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1323.09it/s]
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 8s [Retry 4/5].


Loaded 3 Stage 1 models and 2 Stage 2 models.
Stage 2 validation cutoffs: {'Stage 2 logistic (text + numeric)': 0.8947203886773488, 'Stage 2 LightGBM (numeric)': 0.9122831304450195}


## Provide the input

Choose sentence mode and enter one message at the prompt, or choose text-file mode and set the path. Conversation files should contain one message per line. Prefix lines with speaker names when available, for example `Speaker A: ...`. Do not include timestamps; this notebook has no timestamp parser.

Without speaker prefixes, Stage 2 can still calculate scores, but speaker behavior features are weak approximations. A single sentence is especially out of distribution for conversation-level models.

In [3]:
INPUT_MODE = "text_file"  # Choose "sentence" or "text_file".
INPUT_FILE = WORKSPACE / "user_conversation.txt"

def parse_input_lines(mode, file_path):
    if mode == "sentence":
        sentence = input("Enter one message to score: ").strip()
        raw_lines = [sentence] if sentence else []
    elif mode == "text_file":
        path = Path(file_path)
        if not path.is_absolute():
            path = WORKSPACE / path
        if not path.exists():
            raise FileNotFoundError(f"Input text file not found: {path}")
        raw_lines = path.read_text(encoding="utf-8-sig").splitlines()
    else:
        raise ValueError("INPUT_MODE must be 'sentence' or 'text_file'.")

    messages = []
    for line_number, raw_line in enumerate(raw_lines, start=1):
        line = raw_line.strip()
        if not line:
            continue
        match = re.match(r"^([^:\r\n]{1,60}):\s*(.+)$", line)
        if match and not match.group(1).strip().isdigit():
            speaker = match.group(1).strip()
            text = match.group(2).strip()
            has_speaker = True
        else:
            speaker = "Unknown"
            text = line
            has_speaker = False
        if text:
            messages.append({"message_number": len(messages) + 1, "source_line": line_number, "speaker": speaker, "text": text, "speaker_was_explicit": has_speaker})

    if not messages:
        raise ValueError("No non-empty message lines were found.")
    return pd.DataFrame(messages)

input_messages = parse_input_lines(INPUT_MODE, INPUT_FILE)
explicit_speaker_share = float(input_messages["speaker_was_explicit"].mean())
has_explicit_speakers = explicit_speaker_share == 1.0 and input_messages["speaker"].nunique() >= 2
if not has_explicit_speakers:
    print("Caution: two or more explicit speakers were not identified; Stage 2 behavior-based scores may be unreliable.")
print(f"Loaded {len(input_messages)} message(s). Message text is not printed or saved by this notebook.")

Loaded 4 message(s). Message text is not printed or saved by this notebook.


In [4]:
def build_context_text(frame):
    texts = frame["text"].fillna("").astype(str).tolist()
    context_texts = []
    for index, target in enumerate(texts):
        previous_text = texts[index - 1] if index > 0 else ""
        next_text = texts[index + 1] if index + 1 < len(texts) else ""
        context_texts.append(f"previous: {previous_text} target: {target} next: {next_text}")
    return context_texts

input_messages["model_text"] = build_context_text(input_messages)

def ordered_stage_probabilities(model, model_inputs):
    raw_probabilities = model.predict_proba(model_inputs)
    class_names = [str(value) for value in model.classes_]
    missing_stages = set(STAGE_ORDER) - set(class_names)
    if missing_stages:
        raise ValueError(f"Stage model is missing expected classes: {sorted(missing_stages)}")
    return raw_probabilities[:, [class_names.index(stage) for stage in STAGE_ORDER]]

if "embedding_model" not in globals():
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError("Install sentence-transformers to use the saved Stage 1 embedding model.") from exc
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

stage_probabilities = {}
for model_name, model in stage1_models.items():
    stage_probabilities[model_name] = ordered_stage_probabilities(model, input_messages["model_text"])

embedding_vectors = embedding_model.encode(
    input_messages["model_text"].tolist(),
    batch_size=32,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
stage_probabilities["Sentence embeddings"] = ordered_stage_probabilities(embedding_classifier, embedding_vectors)

def viterbi_decode(emissions, transitions):
    epsilon = np.finfo(float).eps
    log_emissions = np.log(np.clip(emissions, epsilon, 1.0))
    log_transitions = np.log(np.clip(transitions, epsilon, 1.0))
    scores = np.full_like(log_emissions, -np.inf)
    backpointers = np.zeros(log_emissions.shape, dtype=int)
    scores[0] = log_emissions[0]
    for position in range(1, len(log_emissions)):
        candidates = scores[position - 1][:, None] + log_transitions
        backpointers[position] = np.argmax(candidates, axis=0)
        scores[position] = log_emissions[position] + candidates.max(axis=0)
    path = np.zeros(len(log_emissions), dtype=int)
    path[-1] = int(np.argmax(scores[-1]))
    for position in range(len(path) - 1, 0, -1):
        path[position - 1] = backpointers[position, path[position]]
    return path

transition_frame = pd.read_csv(REPORT_DIR / "stage1_transition_matrix.csv", index_col=0)
transition_frame.index = transition_frame.index.map(str)
transition_frame.columns = transition_frame.columns.map(str)
transition_matrix = transition_frame.loc[STAGE_ORDER, STAGE_ORDER].to_numpy(dtype=float)

baseline_probabilities = stage_probabilities["TF-IDF baseline"]
smoothed_path = viterbi_decode(baseline_probabilities, transition_matrix)

stage_rows = []
for model_name, probabilities in stage_probabilities.items():
    predicted_indices = probabilities.argmax(axis=1)
    for row_index, message in input_messages.iterrows():
        probability_row = probabilities[row_index]
        stage_rows.append({
            "message_number": int(message["message_number"]),
            "model": model_name,
            "predicted_stage": STAGE_ORDER[int(predicted_indices[row_index])],
            "probability_200": float(probability_row[0]),
            "probability_600": float(probability_row[1]),
            "probability_900": float(probability_row[2]),
        })

stage_results = pd.DataFrame(stage_rows)
stage_summary = (
    stage_results.pivot(index="message_number", columns="model", values="predicted_stage")
    .reset_index()
)
stage_summary["TF-IDF baseline (Viterbi-smoothed)"] = [STAGE_ORDER[index] for index in smoothed_path]
display(stage_summary)
display(stage_results)

model,message_number,Sentence embeddings,TF-IDF + lexicons,TF-IDF baseline,TF-IDF baseline (Viterbi-smoothed)
0,1,200,200,200,200
1,2,200,200,200,200
2,3,200,200,200,200
3,4,600,200,200,200


,message_number,model,predicted_stage,probability_200,probability_600,probability_900
0,1,TF-IDF baseline,200,0.688466,0.278759,0.032775
1,2,TF-IDF baseline,200,0.643410,0.325164,0.031426
2,3,TF-IDF baseline,200,0.682512,0.287620,0.029868
3,4,TF-IDF baseline,200,0.619765,0.328101,0.052134
4,1,TF-IDF + lexicons,200,0.862050,0.106200,0.031750
5,2,TF-IDF + lexicons,200,0.853479,0.111549,0.034972
6,3,TF-IDF + lexicons,200,0.861131,0.105945,0.032924
7,4,TF-IDF + lexicons,200,0.653775,0.288120,0.058106
8,1,Sentence embeddings,200,0.868180,0.112450,0.019370
9,2,Sentence embeddings,200,0.841056,0.120047,0.038897


## Stage 2 conversation scores and traffic-light summary

The Stage 2 feature construction matches notebook 09 where possible. Since the input format contains no timestamps, time-based features are left missing and handled by each saved model's fitted imputer. The Stage 2 models were trained on PAN-12 conversations; single-message predictions and unlabelled-speaker files are not validated use cases.

In [5]:
stage1_for_stage2 = baseline_probabilities
baseline_classes = [str(value) for value in stage1_models["TF-IDF baseline"].classes_]
baseline_predictions = np.array(STAGE_ORDER)[stage1_for_stage2.argmax(axis=1)]

message_texts = input_messages["text"].fillna("").astype(str)
message_lengths = message_texts.str.len().to_numpy(dtype=float)
speaker_counts = input_messages["speaker"].value_counts()
number_of_messages = len(input_messages)

def first_onset(probabilities, stage_index):
    positions = np.flatnonzero(probabilities[:, stage_index] >= 0.50)
    return float((positions[0] + 1) / number_of_messages) if len(positions) else np.nan

numeric_features = {
    "message_count": float(number_of_messages),
    "mean_message_length": float(message_lengths.mean()),
    "std_message_length": float(message_lengths.std(ddof=0)),
    "author_count": float(input_messages["speaker"].nunique()),
    "dominant_author_share": float(speaker_counts.max() / number_of_messages),
    "question_share": float(message_texts.str.contains(r"\?", regex=True).mean()),
    "fraction_predicted_200": float(np.mean(baseline_predictions == "200")),
    "fraction_predicted_600": float(np.mean(baseline_predictions == "600")),
    "fraction_predicted_900": float(np.mean(baseline_predictions == "900")),
    "stage_600_threshold_onset": first_onset(stage1_for_stage2, 1),
    "stage_900_threshold_onset": first_onset(stage1_for_stage2, 2),
    "mean_stage_600_probability": float(stage1_for_stage2[:, 1].mean()),
    "mean_stage_900_probability": float(stage1_for_stage2[:, 2].mean()),
    "max_stage_600_probability": float(stage1_for_stage2[:, 1].max()),
    "max_stage_900_probability": float(stage1_for_stage2[:, 2].max()),
    "night_time_share": np.nan,
    "duration_minutes": np.nan,
}

conversation_text = " ".join(message_texts.tolist())
stage_columns = [
    "fraction_predicted_200", "fraction_predicted_600", "fraction_predicted_900",
    "stage_600_threshold_onset", "stage_900_threshold_onset",
    "mean_stage_600_probability", "mean_stage_900_probability",
    "max_stage_600_probability", "max_stage_900_probability",
]
behavioral_columns = [
    "message_count", "mean_message_length", "std_message_length", "author_count",
    "dominant_author_share", "question_share", "night_time_share", "duration_minutes",
]
feature_frame = pd.DataFrame([{**numeric_features, "conversation_text": conversation_text}])
numeric_columns = stage_columns + behavioral_columns

risk_rows = []
for model_name, model in stage2_models.items():
    if model_name == "Stage 2 logistic (text + numeric)":
        model_input = feature_frame
    else:
        model_input = feature_frame[numeric_columns]
    risk_score = float(model.predict_proba(model_input)[0, 1])
    cutoff = float(stage2_thresholds[model_name])
    status = "RED" if risk_score >= cutoff else "GREEN"
    risk_rows.append({
        "model": model_name,
        "score": risk_score,
        "red_cutoff": cutoff,
        "individual_status": status,
    })

risk_results = pd.DataFrame(risk_rows)
if (risk_results["individual_status"] == "RED").any():
    overall_status = "RED"
else:
    overall_status = "GREEN"

colors = {"RED": "🔴", "GREEN": "🟢"}
print(f"OVERALL REVIEW TRIAGE: {colors[overall_status]} {overall_status}")
display(risk_results)
if overall_status == "GREEN":
    print("Green means only that both models scored below the heuristic low-score boundary; it is not evidence that the conversation is safe.")
if number_of_messages == 1:
    print("Caution: the conversation-level score is based on one message and should not be treated as a reliable conversation assessment.")
if not has_explicit_speakers:
    print("Caution: speaker-behavior features are inferred from incomplete or absent speaker labels.")
print("Stage 1 predictions are stage labels, not safety labels. They did not determine the traffic-light color.")

print("No user conversation text or score table has been written to a file.")

OVERALL REVIEW TRIAGE: 🟢 GREEN


,model,score,red_cutoff,individual_status
0,Stage 2 logistic (text + numeric),0.042538,0.894720,GREEN
1,Stage 2 LightGBM (numeric),0.046570,0.912283,GREEN


Green means only that both models scored below the heuristic low-score boundary; it is not evidence that the conversation is safe.
Stage 1 predictions are stage labels, not safety labels. They did not determine the traffic-light color.
No user conversation text or score table has been written to a file.
